In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### XGBoost + Optuna + MLflow Pipeline

In [2]:
#Load datasets

train_day_df = pd.read_csv("D:/VS Code Projects/Datasets/Bike Sharing/data_processed/feature_engineered_train_day.csv")
valid_day_df = pd.read_csv("D:/VS Code Projects/Datasets/Bike Sharing/data_processed/feature_engineered_valid_day.csv")
train_hour_df = pd.read_csv("D:/VS Code Projects/Datasets/Bike Sharing/data_processed/feature_engineered_train_hour.csv")
valid_hour_df = pd.read_csv("D:/VS Code Projects/Datasets/Bike Sharing/data_processed/feature_engineered_valid_hour.csv")

In [3]:
### Ensure datetime & remove the column 'Unnamed: 0'
datasets_list =  [train_day_df,train_hour_df,valid_day_df,valid_hour_df]
for dataset in datasets_list:
    dataset['date'] = pd.to_datetime(dataset['date'], errors= 'coerce')
    dataset.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

print('ISSUE FIXED')
train_day_df.dtypes

ISSUE FIXED


date                    datetime64[ns]
month                            int64
weather_situation                int64
feels_like_temp_norm           float64
temp_feel_diff                 float64
wind_speed                     float64
temp_x_wind_speed              float64
casual                           int64
registered                       int64
num_rentals                      int64
dtype: object

In [4]:
target = 'num_rentals'
X_day_train = train_day_df.drop(columns= [target, 'date'])
y_day_train = train_day_df[target]

X_day_valid = valid_day_df.drop(columns= [target, 'date'])
y_day_valid =valid_day_df[target]


X_hour_train = train_hour_df.drop(columns= [target, 'date'])
y_hour_train = train_hour_df[target]

X_hour_valid = valid_hour_df.drop(columns= [target, 'date'])
y_hour_valid =valid_hour_df[target]

#### Optimise Hyperparameters with Optuna

In [ ]:
def objective_day(trial):
    # define hyperparameter search space
    # train XGBoost on day dataset
    # compute RMSE and R²
    # log params and metrics to MLflow
    
    params = {
        # Core structure
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.3, log=True),
        
        # Overfitting control
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        
        # Randomness
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
        
        # Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        
        # Fixed parameters
        "random_state": 19,
        "n_jobs": -1,
        "tree_method": "hist"
    }

    with mlflow.start_run(run_name="trial_day"):
        model = XGBRegressor(**params)
        model.fit(X_day_train, y_day_train)
        
        y_pred = model.predict(X_day_valid)
        rmse = np.sqrt(mean_squared_error(y_day_valid, y_pred))
        r2 = r2_score(y_day_valid, y_pred)
        
        # Log hyperparameters and metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "r2": r2})
    
    return rmse

In [6]:
def objective_hour(trial):
    params = {
        # Core structure
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.3, log=True),
        
        # Overfitting control
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        
        # Randomness
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
        
        # Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        
        # Fixed parameters
        "random_state": 19,
        "n_jobs": -1,
        "tree_method": "hist"
    }

    with mlflow.start_run(run_name="trial_hour"):
        model = XGBRegressor(**params)
        model.fit(X_hour_train, y_hour_train)
        
        y_pred = model.predict(X_hour_valid)
        rmse = np.sqrt(mean_squared_error(y_hour_valid, y_pred))
        r2 = r2_score(y_hour_valid, y_pred)
        
        # Log hyperparameters and metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "r2": r2})
    
    return rmse

In [ ]:
## 1️ Setup MLflow
mlflow.set_tracking_uri(r"file:D:\VS Code Projects\Bike-rental-count-prediction\mlruns")

In [8]:
mlflow.set_experiment("xgboost_optuna_day")
study_day = optuna.create_study(
    study_name="xgb_day_optimisation",
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study_day.optimize(objective_day, n_trials=50)

print("Best RMSE (day):", study_day.best_value)
print("Best params (day):", study_day.best_trial.params)

c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
[I 2026-01-22 23:45:44,371] A new study created in memory with name: xgb_day_optimisation
[I 2026-01-22 23:45:44,524] Trial 0 finished with value: 1188.8243667590264 and parameters: {'n_estimators': 217, 'max_depth': 4, 'learning_rate': 0.09196761223792932, 'min_child_weight': 5, 'gamma': 1.6034298729293868, 'subsample': 0.8223276736172509, 'colsample_bytree': 0.8382891674662865, 'reg_alpha': 0.006836240237222135, 'reg_lambda': 0.

Best RMSE (day): 992.515711462544
Best params (day): {'n_estimators': 246, 'max_depth': 3, 'learning_rate': 0.12940905583300322, 'min_child_weight': 1, 'gamma': 0.35725883982335577, 'subsample': 0.9503474732014198, 'colsample_bytree': 0.8757670037671056, 'reg_alpha': 1.6138282788753846, 'reg_lambda': 2.550572435003872e-06}


In [9]:
mlflow.set_experiment("xgboost_optuna_hour")
study_hour = optuna.create_study(
    study_name="xgb_hour_optimisation",
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study_hour.optimize(objective_hour, n_trials=50)

print("Best RMSE (hour):", study_hour.best_value)
print("Best params (hour):", study_hour.best_trial.params)

2026/01/22 23:45:48 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_hour' does not exist. Creating a new experiment.
[I 2026-01-22 23:45:48,995] A new study created in memory with name: xgb_hour_optimisation
[I 2026-01-22 23:45:49,128] Trial 0 finished with value: 38.84042341698836 and parameters: {'n_estimators': 202, 'max_depth': 5, 'learning_rate': 0.22896334070562346, 'min_child_weight': 1, 'gamma': 1.5109629281358075, 'subsample': 0.9512619087877012, 'colsample_bytree': 0.8076235903432375, 'reg_alpha': 3.189100522381956e-05, 'reg_lambda': 0.025142272270030495}. Best is trial 0 with value: 38.84042341698836.
[I 2026-01-22 23:45:49,282] Trial 1 finished with value: 38.496169655239726 and parameters: {'n_estimators': 319, 'max_depth': 5, 'learning_rate': 0.09085961850310917, 'min_child_weight': 1, 'gamma': 0.6153814297623847, 'subsample': 0.6100118789777926, 'colsample_bytree': 0.8000167074305946, 'reg_alpha': 8.40000591965987e-07, 'reg_lambda': 0.020152976127947222

Best RMSE (hour): 33.801649570278
Best params (hour): {'n_estimators': 243, 'max_depth': 3, 'learning_rate': 0.13942443493239845, 'min_child_weight': 1, 'gamma': 0.8285569162961217, 'subsample': 0.7231405085238518, 'colsample_bytree': 0.9016996061376332, 'reg_alpha': 0.018721440098794083, 'reg_lambda': 0.004797855449974764}


#### Train Final Models with Best Parameters

In [10]:
best_params_day = study_day.best_trial.params
model_day = XGBRegressor(**best_params_day)
model_day.fit(X_day_train, y_day_train)

y_day_pred = model_day.predict(X_day_valid)
rmse_day = np.sqrt(mean_squared_error(y_day_valid, y_day_pred))
r2_day = r2_score(y_day_valid, y_day_pred)

print("Final Day Model Results:")
print("RMSE:", rmse_day)
print("R2:", r2_day)

with mlflow.start_run(run_name="final_day_model"):
    mlflow.log_params(best_params_day)
    mlflow.log_metrics({"rmse": rmse_day, "r2": r2_day})
    mlflow.sklearn.log_model(model_day, "xgb_day_model")


2026/01/22 23:45:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final Day Model Results:
RMSE: 976.9054777715191
R2: 0.6753993034362793


In [11]:
best_params_hour = study_hour.best_trial.params
model_hour = XGBRegressor(**best_params_hour)
model_hour.fit(X_hour_train, y_hour_train)

y_hour_pred = model_hour.predict(X_hour_valid)
rmse_hour = np.sqrt(mean_squared_error(y_hour_valid, y_hour_pred))
r2_hour = r2_score(y_hour_valid, y_hour_pred)

print("Final Hour Model Results:")
print("RMSE:", rmse_hour)
print("R2:", r2_hour)

with mlflow.start_run(run_name="final_hour_model"):
    mlflow.log_params(best_params_hour)
    mlflow.log_metrics({"rmse": rmse_hour, "r2": r2_hour})
    mlflow.sklearn.log_model(model_hour, "xgb_hour_model")

2026/01/22 23:45:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final Hour Model Results:
RMSE: 35.885737178055855
R2: 0.9710771441459656
